# Part 2 - Further Model Analysis

In [2]:
#import
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier


In [4]:
# load train and test data
data = np.load("mushroom_train_test.npz", allow_pickle=True)
X_train = data["X_train"]
X_test  = data["X_test"]
y_train = data["y_train"]
y_test  = data["y_test"]
feat_names = data["feat_names"].tolist()

print(f"Loaded shapes: X_train={X_train.shape}, X_test={X_test.shape}")


Loaded shapes: X_train=(48855, 16), X_test=(12214, 16)


In [32]:
#evaluation function (will be reerun for each model)

def evaluate_and_log(model_name, estimator, X_tr=X_train, X_te=X_test):
    y_pred = estimator.predict(X_te)

    #probability scores for ROC
    if hasattr(estimator, "predict_proba"): #if model accepts probability caluclation, use it
        y_proba = estimator.predict_proba(X_te)[:, 1]
    elif hasattr(estimator, "decision_function"): #otherwise calculate distance from descions boundary (semi probability)
        s = estimator.decision_function(X_te)
        s_min, s_max = s.min(), s.max()
        y_proba = (s - s_min) / (s_max - s_min + 1e-12)
    else: #if model doesnt accept either calculation just use the 1/0 prediction
        y_proba = y_pred.astype(float)

    roc  = roc_auc_score(y_test, y_proba) #use the probability to calculate ROC
    #then do other metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)

    return acc, prec, rec, f1, roc

# 1. Logistic Regression
Hyperparameters tested:

- C (inverse regularization) = 0.01, 0.1, 1.0, 10.0
- solvers = liblinear, lbfgs

In [33]:

lr_runs = []
for C in lr_Cs:
    for solver in lr_solvers:
        for max_iter in lr_max_iter:
            lr = LogisticRegression(C=C, solver=solver, max_iter=max_iter, random_state=42)
            lr.fit(X_train, y_train)

            acc, prec, rec, f1, roc = evaluate_and_log(
                model_name=f"LogReg (C={C}, solver={solver})",
                estimator=lr,
                X_tr=X_train,
                X_te=X_test
            )

            lr_runs.append({
                "C": C, "solver": solver, "max_iter": max_iter,
                "Accuracy":acc, "Precision": prec,
                "Recall": rec, "F1": f1, "ROC_AUC": roc
            })

lr_results = pd.DataFrame(lr_runs).sort_values(["Accuracy","F1","ROC_AUC"], ascending=False).reset_index(drop=True)
lr_results


,C,solver,max_iter,Accuracy,Precision,Recall,F1,ROC_AUC
0,0.01,lbfgs,500,0.643933,0.652863,0.765270,0.704612,0.700720
1,0.10,lbfgs,500,0.642869,0.652103,0.764090,0.703668,0.700765
2,10.00,lbfgs,500,0.642459,0.651845,0.763500,0.703268,0.700763
3,1.00,lbfgs,500,0.642378,0.651801,0.763352,0.703180,0.700755
4,1.00,liblinear,500,0.642214,0.651599,0.763500,0.703125,0.700805
5,10.00,liblinear,500,0.642214,0.651637,0.763352,0.703085,0.700793
6,0.10,liblinear,500,0.641886,0.650967,0.764680,0.703256,0.700878
7,0.01,liblinear,500,0.628295,0.638903,0.759369,0.693946,0.699243


## 2. Multi layer perceptron
Hyperparameters tested:

- hidden layer size= (64, 32), (128, 64), (64, 64)
- alpha (ridge regualrization) = 0.0001, 0.001, 0.01

In [34]:
import pandas as pd
from sklearn.neural_network import MLPClassifier

mlp_hls = [(64, 32), (128, 64), (64, 64)]  # two hidden layers: (layer1, layer2)
mlp_alphas = [0.0001, 0.001, 0.01]         # L2 regularization (alpha)
mlp_acts = ["relu"]                        # activation function
mlp_lrs = ["constant"]                     # learning rate schedule

mlp_runs = []

for hls in mlp_hls:
    for alpha in mlp_alphas:
        for act in mlp_acts:
            for lr in mlp_lrs:
                mlp = MLPClassifier(
                    hidden_layer_sizes=hls,
                    alpha=alpha,
                    activation=act,
                    learning_rate=lr,
                    max_iter=500,
                    random_state=42
                )
                mlp.fit(X_train, y_train)

                # evaluate_and_log returns (acc, prec, rec, f1, roc)
                acc, prec, rec, f1, roc = evaluate_and_log(
                    model_name=f"MLP (hls={hls}, alpha={alpha})",
                    estimator=mlp,
                    X_tr=X_train,
                    X_te=X_test
                )

                mlp_runs.append({
                    "hidden_layer_sizes": hls,
                    "alpha": alpha,
                    "activation": act,
                    "learning_rate": lr,
                    "Accuracy": acc,
                    "Precision": prec,
                    "Recall": rec,
                    "F1": f1,
                    "ROC_AUC": roc
                })

# convert to DataFrame and sort by main metrics
mlp_results = (
    pd.DataFrame(mlp_runs)
    .sort_values(["Accuracy", "F1", "ROC_AUC"], ascending=False)
    .reset_index(drop=True)
)

mlp_results


,hidden_layer_sizes,alpha,activation,learning_rate,Accuracy,Precision,Recall,F1,ROC_AUC
0,"(128, 64)",0.0100,relu,constant,0.999918,0.999852,1.000000,0.999926,1.000000
1,"(64, 64)",0.0010,relu,constant,0.999836,0.999852,0.999852,0.999852,0.999991
2,"(64, 64)",0.0100,relu,constant,0.999509,0.999263,0.999852,0.999558,0.999999
3,"(64, 64)",0.0001,relu,constant,0.999181,0.998821,0.999705,0.999263,0.999998
4,"(64, 32)",0.0010,relu,constant,0.992877,0.994677,0.992476,0.993575,0.999349
5,"(64, 32)",0.0100,relu,constant,0.992631,0.989461,0.997344,0.993387,0.999614
6,"(128, 64)",0.0001,relu,constant,0.984280,0.988576,0.983033,0.985797,0.998865
7,"(64, 32)",0.0001,relu,constant,0.981251,0.982182,0.984066,0.983123,0.998183
8,"(128, 64)",0.0010,relu,constant,0.707958,0.726798,0.759073,0.742585,0.786210


## 3. Random Forest
Hyperparameters tested:

- n_estimators = 100, 200, 400
- max_depths = None, 15, 30
- in_split = 2, 5
- min_leaf = 1, 2
- max_features = "sqrt", "log2", None

In [35]:
# =======================================
# Random Forest Evaluation (no rounding)
# =======================================
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

rf_estimators = [100, 200, 400]  # number of trees
rf_max_depths = [None, 15, 30]   # max depth (None = unlimited)
rf_min_split  = [2, 5]           # min samples to split a node
rf_min_leaf   = [1, 2]           # min samples per leaf
rf_max_features = ["sqrt", "log2", None]  # feature subset per split

rf_runs = []

for n in rf_estimators:
    for md in rf_max_depths:
        for mss in rf_min_split:
            for msl in rf_min_leaf:
                for mf in rf_max_features:
                    rf = RandomForestClassifier(
                        n_estimators=n,
                        max_depth=md,
                        min_samples_split=mss,
                        min_samples_leaf=msl,
                        max_features=mf,
                        random_state=42,
                        n_jobs=-1
                    )
                    rf.fit(X_train, y_train)

                    # evaluate_and_log returns (acc, prec, rec, f1, roc)
                    acc, prec, rec, f1, roc = evaluate_and_log(
                        model_name=f"RF (n={n}, depth={md}, split={mss}, leaf={msl}, feat={mf})",
                        estimator=rf,
                        X_tr=X_train,
                        X_te=X_test
                    )

                    rf_runs.append({
                        "n_estimators": n,
                        "max_depth": md,
                        "min_samples_split": mss,
                        "min_samples_leaf": msl,
                        "max_features": mf,
                        "Accuracy": acc,
                        "Precision": prec,
                        "Recall": rec,
                        "F1": f1,
                        "ROC_AUC": roc
                    })

# Create DataFrame (no rounding, full precision)
rf_results = (
    pd.DataFrame(rf_runs)
    .sort_values(["Accuracy", "F1", "ROC_AUC"], ascending=False)
    .reset_index(drop=True)
)

rf_results


,n_estimators,max_depth,min_samples_split,min_samples_leaf,max_features,Accuracy,Precision,Recall,F1,ROC_AUC
0,100,NaN,2,1,sqrt,0.999918,1.000000,0.999852,0.999926,1.000000
1,100,NaN,2,1,log2,0.999918,1.000000,0.999852,0.999926,1.000000
2,100,30.0,2,1,sqrt,0.999918,1.000000,0.999852,0.999926,1.000000
3,100,30.0,2,1,log2,0.999918,1.000000,0.999852,0.999926,1.000000
4,200,NaN,2,1,sqrt,0.999918,1.000000,0.999852,0.999926,1.000000
...,...,...,...,...,...,...,...,...,...,...
103,200,15.0,2,1,None,0.996234,0.998667,0.994541,0.996600,0.999878
104,400,15.0,2,2,None,0.996152,0.998519,0.994541,0.996526,0.999869
105,400,15.0,5,2,None,0.996152,0.998519,0.994541,0.996526,0.999868
106,200,15.0,2,2,None,0.996152,0.998519,0.994541,0.996526,0.999868


## 4. SVM
Hyperparameters tested:

- C = 0.1, 1, 10
- gammas = "scale", "auto"

In [ ]:

import pandas as pd
from sklearn.svm import SVC

svm_Cs = [0.1, 1, 10]          # L2 regularization strength
svm_gammas = ["scale", "auto"] # kernel coefficient options

svm_runs = []

for C in svm_Cs:
    for gamma in svm_gammas:
        svm = SVC(
            kernel="rbf",
            C=C,
            gamma=gamma,
            probability=True,   # needed for ROC_AUC
            random_state=42
        )
        svm.fit(X_train, y_train)

        # evaluate_and_log returns (acc, prec, rec, f1, roc)
        acc, prec, rec, f1, roc = evaluate_and_log(
            model_name=f"SVM (C={C}, gamma={gamma})",
            estimator=svm,
            X_tr=X_train,
            X_te=X_test
        )

        svm_runs.append({
            "C": C,
            "gamma": gamma,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1": f1,
            "ROC_AUC": roc
        })

# Create DataFrame and sort (no rounding)
svm_results = (
    pd.DataFrame(svm_runs)
    .sort_values(["Accuracy", "F1", "ROC_AUC"], ascending=False)
    .reset_index(drop=True)
)

svm_results


## 5. XGBoost
Hyperparameters tested:

- n_estimators = 200, 400
- lr = 0.05, 0.1
- max_depth = 4, 6, 8
- subsample = 0.7, 0.9
- colsample = 0.7, 1.0

In [ ]:

import pandas as pd
from xgboost import XGBClassifier  # ensure XGBoost is installed

xgb_runs = []

if HAVE_XGB:
    xgb_n_estimators = [200, 400]   # number of trees
    xgb_lr = [0.05, 0.1]            # learning rate
    xgb_max_depth = [4, 6, 8]       # max depth of trees
    xgb_subsample = [0.7, 0.9]      # % of data per tree
    xgb_colsample = [0.7, 1.0]      # % of features per tree

    for n in xgb_n_estimators:
        for lr in xgb_lr:
            for md in xgb_max_depth:
                for subs in xgb_subsample:
                    for col in xgb_colsample:
                        xgb = XGBClassifier(
                            n_estimators=n,
                            learning_rate=lr,
                            max_depth=md,
                            subsample=subs,
                            colsample_bytree=col,
                            tree_method="hist",
                            predictor="auto",
                            eval_metric="logloss",
                            random_state=42,
                            n_jobs=-1
                        )
                        xgb.fit(X_train, y_train)

                        # evaluate_and_log returns (acc, prec, rec, f1, roc)
                        acc, prec, rec, f1, roc = evaluate_and_log(
                            model_name=f"XGB (n={n}, lr={lr}, depth={md}, subs={subs}, col={col})",
                            estimator=xgb,
                            X_tr=X_train,
                            X_te=X_test
                        )

                        xgb_runs.append({
                            "n_estimators": n,
                            "learning_rate": lr,
                            "max_depth": md,
                            "subsample": subs,
                            "colsample_bytree": col,
                            "Accuracy": acc,
                            "Precision": prec,
                            "Recall": rec,
                            "F1": f1,
                            "ROC_AUC": roc
                        })
else:
    print("XGBoost not available in this environment.")

if HAVE_XGB:
    xgb_results = (
        pd.DataFrame(xgb_runs)
        .sort_values(["Accuracy", "F1", "ROC_AUC"], ascending=False)
        .reset_index(drop=True)
    )
    xgb_results


## 6. Best Model Analysis

In [ ]:

# combine
model_results_full = pd.concat(
    [df for df in [lr_results, mlp_results, rf_results, svm_results, xgb_results] if not df.empty],
    ignore_index=True
)

# rank by overall sum of metrics
ranked = model_results_full.sort_values("ScoreSum", ascending=False).reset_index(drop=True)


## 7. Further exploration of best model (RF)


## 7.1 performance vs hyperparamter change
we graphed the change in performace as esch hyperparameter changes, the results (graphs below) indicated that performace increased with:
- lower number of esimators
- 

In [ ]:

def plot_param_effect(df, param, metrics=["Accuracy"], figsize=(7,5)):
    plt.figure(figsize=figsize)
    for m in metrics:
        avg_df = df.groupby(param)[m].mean().reset_index()
        plt.plot(avg_df[param], avg_df[m], marker="o", lw=2, label=m)
    plt.title(f"Performance vs {param} (mean across others)")
    plt.xlabel(param)
    plt.ylabel("Score")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.show()

plot_param_effect(rf_results, "n_estimators", metrics=["Accuracy","F1"])
plot_param_effect(rf_results, "max_depth", metrics=["Accuracy","F1"])
plot_param_effect(rf_results, "max_features", metrics=["Accuracy","F1"])
plot_param_effect(rf_results, "min_samples_split", metrics=["Accuracy","F1"])
plot_param_effect(rf_results, "min_samples_leaf", metrics=["Accuracy","F1"])


## 7.2 Test the new findings
we tested new hyperparemters suggested by the old findings

In [ ]:

rf_estimators = [40, 60, 80] #num of trees
rf_max_depths = [None, 15, 30, 45] #max depth of trees (none means they can go to infinity)
rf_min_split = [2, 3] #num of samples at which a node splits (2-> if theres 2 samples in a node -> split)
rf_min_leaf = [1, 2] #min amt of samples that must be in each leaf (the higher the min, the higher the regularization)

rf_runs = []
for n in rf_estimators:
    for md in rf_max_depths:
        for mss in rf_min_split:
            for msl in rf_min_leaf:
                for mf in rf_max_features:
                    rf = RandomForestClassifier(
                        n_estimators=n, max_depth=md,
                        min_samples_split=mss, min_samples_leaf=msl,
                        max_features=mf, random_state=42, n_jobs=-1
                    )
                    rf.fit(X_train, y_train)
                    y_pred = rf.predict(X_test)
                    y_proba = rf.predict_proba(X_test)[:, 1]

                    acc = accuracy_score(y_test, y_pred)
                    prec = precision_score(y_test, y_pred, zero_division=0)
                    rec  = recall_score(y_test, y_pred, zero_division=0)
                    f1   = f1_score(y_test, y_pred, zero_division=0)
                    roc  = roc_auc_score(y_test, y_proba)

                    rf_runs.append({
                        "n_estimators": n, "max_depth": md,
                        "min_samples_split": mss, "min_samples_leaf": msl, "max_features": mf,
                        "Accuracy": round(acc, 4), "Precision": round(prec, 4),
                        "Recall": round(rec, 4), "F1": round(f1, 4), "ROC_AUC": round(roc, 4)
                    })


In [ ]:
# make sure these metric columns exist
metrics = ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]

# compute total (or mean) score per row
rf_results["ScoreSum"] = rf_results[metrics].sum(axis=1)

# sort by that total score (descending = best first)
rf_results = rf_results.sort_values("ScoreSum", ascending=False).reset_index(drop=True)

display(rf_results.head(11))


The results show that the above 11 models all have the bext performance (score sum) of 4.9997 out of 5, we decided to go with the simplest model:

Random forest:

- n_estimators=100

- max_depth=30

- min_samples_split=2

- min_samples_leaf=1 

- max_features="log2"

or model at index 0 in the above table